In [1]:
#Since NCF works mostly to understand user behavior we use only ratings and not plots/genres
import pandas as pd

ratings = pd.read_csv(r"D:\MINI PROJECT\DATASET\MovieLensDataset20M\rating.csv")

ratings = ratings[["userId", "movieId", "rating", "timestamp"]]

ratings.head()


,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [2]:
#encoding users and movies

from sklearn.preprocessing import LabelEncoder

user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

ratings["user_encoded"] = user_encoder.fit_transform(ratings["userId"])
ratings["movie_encoded"] = movie_encoder.fit_transform(ratings["movieId"])

num_users = ratings["user_encoded"].nunique()
num_movies = ratings["movie_encoded"].nunique()

print(num_users, num_movies)


138493 26744


In [3]:
ratings = ratings.sort_values("timestamp")

split_index = int(len(ratings) * 0.8)

train_ratings = ratings.iloc[:split_index]
test_ratings = ratings.iloc[split_index:]

X_train = train_ratings[["user_encoded", "movie_encoded"]]
y_train = train_ratings["rating"]

X_test = test_ratings[["user_encoded", "movie_encoded"]]
y_test = test_ratings["rating"]

print("Train size:", len(train_ratings))
print("Test size:", len(test_ratings))

Train size: 16000210
Test size: 4000053


In [4]:
#building the NCF model
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate
from tensorflow.keras.models import Model


In [5]:
#creating user embeddings
embedding_dim = 50

user_input = Input(shape=(1,))
user_embedding = Embedding(num_users, embedding_dim)(user_input)
user_vec = Flatten()(user_embedding)

movie_input = Input(shape=(1,))
movie_embedding = Embedding(num_movies, embedding_dim)(movie_input)
movie_vec = Flatten()(movie_embedding)


In [6]:
#building neural interaction layers
concat = Concatenate()([user_vec, movie_vec])

dense = Dense(128, activation="relu")(concat)
dense = Dense(64, activation="relu")(dense)
output = Dense(1)(dense)


In [7]:
#building model
model = Model([user_input, movie_input], output)

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 50)     │  6,924,650 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 50)     │  1,337,200 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 50)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 50)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 100)       │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     12,928 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         65 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,283,099 (31.60 MB)

 Trainable params: 8,283,099 (31.60 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.fit(
    [X_train["user_encoded"], X_train["movie_encoded"]],
    y_train,
    epochs=5,
    batch_size=256,
    validation_split=0.1
)


Epoch 1/5
56251/56251 ━━━━━━━━━━━━━━━━━━━━ 1685s 30ms/step - loss: 0.7451 - mae: 0.6654 - val_loss: 0.8405 - val_mae: 0.7018
Epoch 2/5
56251/56251 ━━━━━━━━━━━━━━━━━━━━ 1232s 22ms/step - loss: 0.6590 - mae: 0.6229 - val_loss: 0.8599 - val_mae: 0.7221
Epoch 3/5
56251/56251 ━━━━━━━━━━━━━━━━━━━━ 2565s 46ms/step - loss: 0.6200 - mae: 0.6016 - val_loss: 0.8710 - val_mae: 0.7285
Epoch 4/5
56251/56251 ━━━━━━━━━━━━━━━━━━━━ 3246s 57ms/step - loss: 0.5867 - mae: 0.5828 - val_loss: 0.8947 - val_mae: 0.7452
Epoch 5/5
56251/56251 ━━━━━━━━━━━━━━━━━━━━ 1142s 20ms/step - loss: 0.5590 - mae: 0.5666 - val_loss: 0.8982 - val_mae: 0.7415


In [9]:
# Save the entire model to a file
# The .keras extension is recommended for Keras 3
model.save('ncfmodel.keras') 


PREDICTING USING THE BUILT MODEL

In [10]:
#Building loopkup mappings
# Encoded → original mappings
encoded_to_movieId = dict(
    zip(ratings["movie_encoded"], ratings["movieId"])
)

movieId_to_encoded = dict(
    zip(ratings["movieId"], ratings["movie_encoded"])
)


In [11]:
import joblib

# Save encoders
joblib.dump(user_encoder, "user_encoder.pkl")
joblib.dump(movie_encoder, "movie_encoder.pkl")

print("Encoders saved successfully.")


Encoders saved successfully.


In [12]:
#to distinguish between aldready rated and not yet rated movies for a user
#if this fails we wills till be recommending aldready watched films
def get_seen_movies(user_id, ratings_df):
    return set(
        ratings_df[ratings_df["userId"] == user_id]["movieId"]
    )


In [13]:
#NCF Recommendation function
import numpy as np

def recommend_movies_ncf(
    user_id,
    ncf_model,
    ratings_df,
    movies_df,
    user_encoder,
    movie_encoder,
    top_n=10
):
    # Encode user
    if user_id not in user_encoder.classes_:
        raise ValueError("User not found in training data")

    user_encoded = user_encoder.transform([user_id])[0]

    # Movies user has already rated
    seen_movies = get_seen_movies(user_id, ratings_df)
    known_movie_ids = set(movie_encoder.classes_)

    # Candidate movies = all movies - seen movies
    all_movie_ids = known_movie_ids 
    candidate_movies = [
        m for m in all_movie_ids if m not in seen_movies
    ]

    # Encode candidate movies
    candidate_encoded = movie_encoder.transform(candidate_movies)

    # Prepare model inputs
    user_input = np.full(len(candidate_encoded), user_encoded)
    movie_input = np.array(candidate_encoded)

    # Predict ratings
    predicted_ratings = ncf_model.predict(
        [user_input, movie_input],
        verbose=0
    ).flatten()

    # Rank movies by predicted rating
    top_indices = predicted_ratings.argsort()[::-1][:top_n]
    recommended_movie_ids = [
        candidate_movies[i] for i in top_indices
    ]

    # Return movie details
    return movies_df[
        movies_df["movieId"].isin(recommended_movie_ids)
    ][["movieId", "title", "genres"]]


In [14]:
#Testing the recommnder
movies=pd.read_csv("D:\MINI PROJECT\DATASET\MovieLensDataset20M\movie.csv")

user_id = 10  # example user
ncf_recommendations = recommend_movies_ncf(
    user_id=user_id,
    ncf_model=model,
    ratings_df=train_ratings,
    movies_df=movies,
    user_encoder=user_encoder,
    movie_encoder=movie_encoder,
    top_n=10
)

ncf_recommendations


,movieId,title,genres
49,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
293,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
315,318,"Shawshank Redemption, The (1994)",Crime|Drama
587,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
602,608,Fargo (1996),Comedy|Crime|Drama|Thriller
902,919,"Wizard of Oz, The (1939)",Adventure|Children|Fantasy|Musical
1169,1193,One Flew Over the Cuckoo's Nest (1975),Drama
1187,1213,Goodfellas (1990),Crime|Drama
2772,2858,American Beauty (1999),Comedy|Drama
4132,4226,Memento (2000),Mystery|Thriller


In [15]:
# Returns the movie_id column for rows where user_id is 10
user10=ratings.loc[ratings['userId'] == 10, 'movieId']

movie=pd.read_csv("D:\MINI PROJECT\DATASET\MovieLensDataset20M\movie.csv")
for i in user10:
    print(movie.loc[movie["movieId"]==i,"title"])


1179    Lawrence of Arabia (1962)
Name: title, dtype: object
523    Schindler's List (1993)
Name: title, dtype: object
352    Forrest Gump (1994)
Name: title, dtype: object
895    Casablanca (1942)
Name: title, dtype: object
1944    Saving Private Ryan (1998)
Name: title, dtype: object
1171    Star Wars: Episode V - The Empire Strikes Back...
Name: title, dtype: object
1222    Bridge on the River Kwai, The (1957)
Name: title, dtype: object
1876    Last Emperor, The (1987)
Name: title, dtype: object
1182    Apocalypse Now (1979)
Name: title, dtype: object
1196    Full Metal Jacket (1987)
Name: title, dtype: object
1214    Glory (1989)
Name: title, dtype: object
1072    Crying Game, The (1992)
Name: title, dtype: object
3020    Backdraft (1991)
Name: title, dtype: object
1974    Negotiator, The (1998)
Name: title, dtype: object
2444    Planet of the Apes (1968)
Name: title, dtype: object
1173    Raiders of the Lost Ark (Indiana Jones and the...
Name: title, dtype: object
257    Star Wars